> **repo 안내**: 4-모델 앙상블(P8/P10/P11/P12, 'LLM 앙상블')의 랭킹을 **채점표(정답 기준)**로 두고,
> P7(Hybrid 리랭커)과 KeyBART 베이스라인 중 어느 쪽이 채점표 순서를 더 잘 재현하는지 채점한다.
>
> **바로 실행됨**: 팀 원본 랭킹 시트(`../data/keyphrase_ranking_raw.csv`, `Keyphrase_최종` 내보내기)가
> 있으면 전체 파이프라인을 돌리고, **없으면 저장소에 포함된 요약 결과를 불러와 표시**한다(무거운
> 임베딩·다운로드 skip). 해설: `../docs/ENSEMBLE_REFERENCE_COMPARISON.md`.


# 키프레이즈 랭킹 비교: Hybrid vs KeyBART (LLM 앙상블 기준)

LLM 앙상블의 랭킹을 정답(ground truth)으로 두고, Hybrid 모델과 KeyBART 모델 중
어느 쪽 랭킹이 LLM 순서를 더 잘 따라가는지 비교합니다.

**파이프라인**
1. CSV 파싱 (문서 단위 구조화)
2. 키워드 임베딩 (SentenceTransformer)
3. LLM ↔ 후보(Hybrid/KeyBART) 키워드 1:1 매칭
4. 랭킹 지표 계산 (nDCG@10, RBO@10)
5. 통계 검정 (Wilcoxon signed-rank test)
6. CSV 출력 (문서별 결과 + 요약)

> Colab 상단 메뉴에서 `런타임 > 런타임 유형 변경 > GPU` 로 설정하고 실행하세요.


## 1. 라이브러리 설치

In [1]:
# 의존성은 ../requirements.txt 로 설치 (원본 파이프라인은 sentence-transformers/scipy 사용)


## 2. 데이터 업로드
아래 셀 실행 시 뜨는 업로드 창에서 `KeyBART_Hybrid_LLM_비교.csv` 파일을 선택하세요.


In [2]:
import os
from pathlib import Path
# 팀 원본 랭킹 시트 경로 (문서별 LLM/Hybrid/KeyBART top-10). 없으면 요약 결과로 대체.
RAW_PATH = '../data/keyphrase_ranking_raw.csv'
HAVE_RAW = os.path.exists(RAW_PATH)
print('원본 시트 있음 → 전체 파이프라인 실행' if HAVE_RAW
      else '원본 시트 없음 → 저장소의 요약 결과를 표시합니다 (맨 아래 셀)')


원본 시트 없음 → 저장소의 요약 결과를 표시합니다 (맨 아래 셀)


## 3. CSV 파싱

원본 CSV는 3줄의 헤더(제목, 그룹명, 컬럼명) 뒤에 실제 데이터가 이어지는 구조입니다.
각 문서(id)마다 LLM / Hybrid / KeyBART 세 가지 top-10 키워드 리스트를 rank 순으로 복원합니다.


In [3]:
if HAVE_RAW:  # 원본 랭킹 시트가 있을 때만 전체 파이프라인 실행
    import pandas as pd

    raw = pd.read_csv(RAW_PATH, skiprows=3, header=None, encoding="utf-8")
    raw = raw.iloc[:, :8]
    raw.columns = [
        "id", "title",
        "rank_llm", "keyphrase_llm",
        "rank_hybrid", "keyphrase_hybrid",
        "rank_keybart", "keyphrase_keybart",
    ]
    raw = raw.dropna(subset=["id"])

    docs = {}
    for doc_id, g in raw.groupby("id", sort=False):
        docs[doc_id] = {
            "title": g["title"].iloc[0],
            "llm": list(g.sort_values("rank_llm")["keyphrase_llm"]),
            "hybrid": list(g.sort_values("rank_hybrid")["keyphrase_hybrid"]),
            "keybart": list(g.sort_values("rank_keybart")["keyphrase_keybart"]),
        }

    print(f"파싱된 문서 수: {len(docs)}")
    sample_id = next(iter(docs))
    docs[sample_id]

## 4. 키워드 임베딩

문서 내 등장 여부와 무관하게 "같은 개념"인지 판단하려면 표면적 문자열이 아니라
의미(semantic) 기준으로 비교해야 합니다. 전체 데이터에서 등장하는 고유 키워드를 모아
한 번에 임베딩합니다 (중복 인코딩 방지 → 속도 최적화).


In [4]:
if HAVE_RAW:  # 원본 랭킹 시트가 있을 때만 전체 파이프라인 실행
    from sentence_transformers import SentenceTransformer
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

    def normalize(text):
        return str(text).strip().lower()

    all_phrases = set()
    for d in docs.values():
        all_phrases.update(normalize(p) for p in d["llm"])
        all_phrases.update(normalize(p) for p in d["hybrid"])
        all_phrases.update(normalize(p) for p in d["keybart"])
    all_phrases = sorted(all_phrases)
    print("고유 키워드 수:", len(all_phrases))

    embeddings = model.encode(
        all_phrases, batch_size=256, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    )
    phrase2emb = {p: e for p, e in zip(all_phrases, embeddings)}

## 5. 매칭 함수 + 랭킹 지표

**매칭**: 문서마다 LLM의 top-10과 후보 모델의 top-10 사이에서, 유사도 행렬을 만들고
`linear_sum_assignment`로 총 유사도가 최대가 되는 1:1 매칭을 찾습니다.
유사도가 임계값(threshold) 미만이면 매칭하지 않은 것으로 처리합니다 (같은 개념이 아니라고 판단).

**nDCG@10**: LLM 순위를 정답 관련도(relevance)로 사용합니다. LLM 1위 키워드의 관련도가 가장 높습니다.

$$
rel_i = 11 - \text{rank}_{LLM}(i), \qquad
DCG@10 = \sum_{i \in \text{matched}} \frac{rel_i}{\log_2(\text{rank}_{cand}(i)+1)}, \qquad
nDCG@10 = \frac{DCG@10}{IDCG@10}
$$

**RBO@10 (Rank-Biased Overlap)**: 깊이 $d$까지의 겹침 비율을 상위일수록 크게 가중합니다.

$$
RBO = (1-p)\sum_{d=1}^{10} p^{\,d-1} \cdot \frac{|\text{matched pairs with rank} \le d|}{d}
$$


In [5]:
if HAVE_RAW:  # 원본 랭킹 시트가 있을 때만 전체 파이프라인 실행
    import numpy as np
    from scipy.optimize import linear_sum_assignment

    SIM_THRESHOLD = 0.75   # "같은 개념"으로 볼 유사도 기준 (필요시 조정)
    RBO_P = 0.9            # RBO의 persistence 파라미터 (클수록 하위 순위까지 반영)

    IDEAL_RELEVANCE = list(range(10, 0, -1))
    IDCG10 = sum(r / np.log2(i + 2) for i, r in enumerate(IDEAL_RELEVANCE))

    def match_pairs(llm_list, cand_list):
        n, m = len(llm_list), len(cand_list)
        sim = np.zeros((n, m))
        for i, lp in enumerate(llm_list):
            for j, cp in enumerate(cand_list):
                if lp == cp:
                    sim[i, j] = 1.0
                else:
                    sim[i, j] = float(np.dot(phrase2emb[lp], phrase2emb[cp]))
        row_idx, col_idx = linear_sum_assignment(-sim)
        pairs = {}
        for i, j in zip(row_idx, col_idx):
            if sim[i, j] >= SIM_THRESHOLD:
                pairs[i + 1] = j + 1  # 1-based rank
        return pairs

    def ndcg_score(pairs):
        dcg = 0.0
        for llm_rank, cand_rank in pairs.items():
            rel = 11 - llm_rank
            dcg += rel / np.log2(cand_rank + 1)
        return dcg / IDCG10

    def rbo_score(pairs, p=RBO_P, k=10):
        total = 0.0
        for d in range(1, k + 1):
            overlap = sum(1 for lr, cr in pairs.items() if lr <= d and cr <= d)
            total += (p ** (d - 1)) * (overlap / d)
        return (1 - p) * total

## 6. 전체 문서에 적용

In [6]:
if HAVE_RAW:  # 원본 랭킹 시트가 있을 때만 전체 파이프라인 실행
    from tqdm import tqdm

    results = []
    for doc_id, d in tqdm(docs.items()):
        llm_list = [normalize(p) for p in d["llm"]]
        hybrid_list = [normalize(p) for p in d["hybrid"]]
        keybart_list = [normalize(p) for p in d["keybart"]]

        pairs_h = match_pairs(llm_list, hybrid_list)
        pairs_k = match_pairs(llm_list, keybart_list)

        ndcg_h, ndcg_k = ndcg_score(pairs_h), ndcg_score(pairs_k)
        rbo_h, rbo_k = rbo_score(pairs_h), rbo_score(pairs_k)

        results.append({
            "id": doc_id,
            "title": d["title"],
            "nDCG_hybrid": ndcg_h,
            "nDCG_keybart": ndcg_k,
            "RBO_hybrid": rbo_h,
            "RBO_keybart": rbo_k,
            "matched_hybrid": len(pairs_h),
            "matched_keybart": len(pairs_k),
            "better_by_ndcg": "Hybrid" if ndcg_h > ndcg_k else ("KeyBART" if ndcg_k > ndcg_h else "Tie"),
            "better_by_rbo": "Hybrid" if rbo_h > rbo_k else ("KeyBART" if rbo_k > rbo_h else "Tie"),
        })

    result_df = pd.DataFrame(results)
    result_df.head()

## 7. 통계 검정 + 요약

In [7]:
if HAVE_RAW:  # 원본 랭킹 시트가 있을 때만 전체 파이프라인 실행
    from scipy.stats import wilcoxon

    stat_ndcg, p_ndcg = wilcoxon(result_df["nDCG_hybrid"], result_df["nDCG_keybart"])
    stat_rbo, p_rbo = wilcoxon(result_df["RBO_hybrid"], result_df["RBO_keybart"])

    summary_df = pd.DataFrame({
        "metric": ["nDCG@10", "RBO@10"],
        "mean_hybrid": [result_df["nDCG_hybrid"].mean(), result_df["RBO_hybrid"].mean()],
        "mean_keybart": [result_df["nDCG_keybart"].mean(), result_df["RBO_keybart"].mean()],
        "win_rate_hybrid": [
            (result_df["better_by_ndcg"] == "Hybrid").mean(),
            (result_df["better_by_rbo"] == "Hybrid").mean(),
        ],
        "win_rate_keybart": [
            (result_df["better_by_ndcg"] == "KeyBART").mean(),
            (result_df["better_by_rbo"] == "KeyBART").mean(),
        ],
        "wilcoxon_stat": [stat_ndcg, stat_rbo],
        "p_value": [p_ndcg, p_rbo],
    })
    summary_df

## 8. 결과 저장 (CSV 출력)
- `keyphrase_ranking_comparison_per_doc.csv`: 문서별 상세 결과
- `keyphrase_ranking_comparison_summary.csv`: 전체 요약 + 통계 검정 결과


In [8]:
if HAVE_RAW:  # 원본 랭킹 시트가 있을 때만 전체 파이프라인 실행
    result_df.to_csv("keyphrase_ranking_comparison_per_doc.csv", index=False, encoding="utf-8-sig")
    summary_df.to_csv("keyphrase_ranking_comparison_summary.csv", index=False, encoding="utf-8-sig")
    print("저장 완료")

## 9. (선택) 분포 시각화

In [9]:
if HAVE_RAW:  # 원본 랭킹 시트가 있을 때만 전체 파이프라인 실행
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].boxplot([result_df["nDCG_hybrid"], result_df["nDCG_keybart"]], labels=["Hybrid", "KeyBART"])
    axes[0].set_title("nDCG@10 Distribution")
    axes[1].boxplot([result_df["RBO_hybrid"], result_df["RBO_keybart"]], labels=["Hybrid", "KeyBART"])
    axes[1].set_title("RBO@10 Distribution")
    plt.tight_layout()
    plt.show()

In [10]:
# 채점 결과 요약 — 원본 계산(HAVE_RAW) 또는 저장소에 포함된 요약 CSV
import pandas as pd
from pathlib import Path
root = Path.cwd()
while not (root / 'results').exists() and root != root.parent:
    root = root.parent
if HAVE_RAW:
    summary_out = summary_df
else:
    print('저장소 포함 요약 결과 (원 실행과 동일 — 앙상블 채점표 기준):')
    summary_out = pd.read_csv(root / 'results/metrics/ensemble_reference_comparison_summary.csv')
summary_out


저장소 포함 요약 결과 (원 실행과 동일 — 앙상블 채점표 기준):


,metric,mean_hybrid_P7,mean_keybart_baseline,win_rate_P7,win_rate_keybart,wilcoxon_stat,p_value
0,nDCG@10,0.897826,0.784173,0.85165,0.14835,16340959.0,0.0
1,RBO@10,0.451763,0.325453,0.85060,0.14930,14620044.5,0.0
